In [8]:
import boto3

In [11]:
session = boto3.Session(profile_name='hiv-project')
athena = session.client('athena')

In [9]:
tables = ['pi', 'nrti', 'nnrti', 'ini', 'cai']

In [5]:
tables = ['pi', 'nrti', 'nnrti', 'ini', 'cai']
query_ids = {}

for table in tables:
    sql = f"""
    CREATE EXTERNAL TABLE IF NOT EXISTS hiv_analysis.Genotype_Phenotype_{table}
    ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
    WITH SERDEPROPERTIES (
        'separatorChar' = ',',
        'quoteChar' = '"',
        'escapeChar' = '\\\\'
    )
    STORED AS TEXTFILE
    LOCATION 's3://hiv-data-022784797781/raw/Genotype-Phenotype-{table}/'
    TBLPROPERTIES ('skip.header.line.count'='1')
    """
    response = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={'Database': 'hiv_analysis'},
        ResultConfiguration={'OutputLocation': 's3://hiv-data-022784797781/athena-results/'}
    )
    query_ids[table] = response['QueryExecutionId']
    print(f"{table}: {response['QueryExecutionId']}")

pi: 45b664ea-9775-47db-96b9-33020e2bd990
nrti: 8b404f13-4a5c-419c-ae4f-ae7692f775f8
nnrti: 9443f57b-50ac-45e5-a7fc-2d8baac49b5f
ini: b45531a6-259b-4d97-b92f-2b32919e2216
cai: 3d5c5e38-f293-4475-85ab-78bdb976f39b


In [7]:
for table, query_id in query_ids.items():
    response = athena.get_query_execution(QueryExecutionId=query_id)
    state = response['QueryExecution']['Status']['State']
    reason = response['QueryExecution']['Status'].get('StateChangeReason', '')
    print(f"{table}: {state} - {reason}")

pi: SUCCEEDED - 
nrti: SUCCEEDED - 
nnrti: SUCCEEDED - 
ini: SUCCEEDED - 
cai: SUCCEEDED - 
